In [24]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

In [25]:
# Custom Noise Transform
class AddGaussianNoise(object):
    def __init__(self, mean=0., std=1.):
        self.std = std
        self.mean = mean
    def __call__(self, tensor):
        return tensor + torch.randn(tensor.size()) * self.std + self.mean
    def __repr__(self):
        return self.__class__.__name__ + '(mean={0}, std={1})'.format(self.mean, self.std)

# Transform to tensor and normalize with Augmentation
transform = transforms.Compose([
    # transforms.RandomAffine(degrees=15, scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
    # AddGaussianNoise(0., 0.1)
])

# Load MNIST dataset
train_set = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

In [26]:
# Create DataLoaders
batch_size = 64
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

print(f"Train set size: {len(train_set)}")
print(f"Test set size: {len(test_set)}")

Train set size: 60000
Test set size: 10000


In [27]:
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self):
        super(Encoder, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, stride=2, padding=1)  # 28x28 -> 14x14
        self.conv2 = nn.Conv2d(16, 32, 3, stride=2, padding=1) # 14x14 -> 7x7
        self.conv3 = nn.Conv2d(32, 64, 7)                      # 7x7   -> 1x1
        self.fc = nn.Linear(64, 2)                             # 64 -> 2

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = x.view(x.size(0), -1) 
        x = self.fc(x)
        return x

class Decoder(nn.Module):
    def __init__(self):
        super(Decoder, self).__init__()
        # Code changed to use Upsample and Conv instead of ConvTranspose2d
        # This follows the 'only linear layer to scale up' instruction (Upsample is a linear op)
        self.fc = nn.Linear(2, 64 * 7 * 7)
        
        self.upsample1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv1 = nn.Conv2d(64, 32, 3, padding=1)
        
        self.upsample2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv2 = nn.Conv2d(32, 1, 3, padding=1)

    def forward(self, x):
        x = self.fc(x)
        x = x.view(x.size(0), 64, 7, 7)
        x = F.relu(self.conv1(self.upsample1(x)))
        x = torch.tanh(self.conv2(self.upsample2(x)))
        return x

class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

# Initialize model
model = Autoencoder()

In [30]:
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import torch.optim.lr_scheduler as lr_scheduler

# Loss function and Optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Learning Rate Scheduler
# Decrease by /10 every 5 epochs
scheduler = lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.8)

# TensorBoard Setup (Separated)
# writer = SummaryWriter('runs/mnist_autoencoder_pure')

# Training settings
num_epochs = 20
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Training loop
global_step = 0
for epoch in range(10, num_epochs):
    model.train()
    total_loss = 0
    
    loop = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}', leave=True)
    
    for data in loop:
        img, _ = data
        img = img.to(device)
        
        output = model(img)
        loss = criterion(output, img)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        writer.add_scalar('Training Loss', loss.item(), global_step)
        
        if global_step % 100 == 0:
            with torch.no_grad():
                num_images = 20
                # 15 images
                original_grid = torchvision.utils.make_grid(img[:num_images].cpu(), nrow=5, normalize=True, value_range=(-1, 1))
                recon_grid = torchvision.utils.make_grid(output[:num_images].cpu(), nrow=5, normalize=True, value_range=(-1, 1))
                
                # Resize grids for visibility (scale factor 4)
                original_grid = F.interpolate(original_grid.unsqueeze(0), scale_factor=4, mode='nearest').squeeze(0)
                recon_grid = F.interpolate(recon_grid.unsqueeze(0), scale_factor=4, mode='nearest').squeeze(0)

                writer.add_image('Original Images', original_grid, global_step)
                writer.add_image('Reconstructed Images', recon_grid, global_step)

        loop.set_postfix(loss=loss.item(), lr=optimizer.param_groups[0]['lr'])
        global_step += 1
    
    scheduler.step()
    print(f'Epoch [{epoch+1}/{num_epochs}], Average Loss: {total_loss/len(train_loader):.4f}, LR: {optimizer.param_groups[0]["lr"]}')

writer.close()

Epoch 11/20: 100%|██████████| 938/938 [00:29<00:00, 31.49it/s, loss=0.159, lr=0.001]


Epoch [11/20], Average Loss: 0.1653, LR: 0.0008


Epoch 12/20: 100%|██████████| 938/938 [00:18<00:00, 51.46it/s, loss=0.174, lr=0.0008]


Epoch [12/20], Average Loss: 0.1643, LR: 0.00064


Epoch 13/20: 100%|██████████| 938/938 [00:51<00:00, 18.36it/s, loss=0.169, lr=0.00064]


...

Epoch [18/20], Average Loss: 0.1614, LR: 0.0001677721600000001


Epoch 19/20: 100%|██████████| 938/938 [00:33<00:00, 28.26it/s, loss=0.152, lr=0.000168]


Epoch [19/20], Average Loss: 0.1611, LR: 0.00013421772800000008


Epoch 20/20: 100%|██████████| 938/938 [01:02<00:00, 14.96it/s, loss=0.152, lr=0.000134]

Epoch [20/20], Average Loss: 0.1610, LR: 0.00010737418240000007


In [29]:
torch.save(model.state_dict(), 'pure_model_weights.pth')